In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import warnings

import numpy as np
import torch
from maite_datasets.image_classification import MNIST

from dataeval.data import Crop, Limit, Resize, SelectChannels, View
from dataeval.extractors import TorchExtractor
from dataeval.flags import ImageStats
from dataeval.quality import Outliers

In [3]:
mnist = View(MNIST("./data", image_set="test", download=True), [Limit(500)])
print("source dataset size:", len(mnist))
print("source image shape:", np.asarray(mnist[0][0]).shape)

source dataset size: 500
source image shape: (1, 28, 28)


In [4]:
deployed = View(mnist, [Resize((14, 14))])
print("deployed image shape:", np.asarray(deployed[0][0]).shape)

deployed image shape: (1, 14, 14)


In [5]:
bordered = View(mnist, [Crop((4, 4, 24, 24))])
print("cropped image shape:", np.asarray(bordered[0][0]).shape)

cropped image shape: (1, 20, 20)


In [6]:
# TEST ASSERTION CELL ###
assert np.asarray(deployed[0][0]).shape == (1, 14, 14)
assert np.asarray(bordered[0][0]).shape == (1, 20, 20)

In [7]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    Outliers(flags=ImageStats.DIMENSION | ImageStats.VISUAL).evaluate(deployed)

invalidated = [w for w in caught if w.category.__name__ == "StatsInvalidatedWarning"]
print(str(invalidated[0].message))

Resize(size=(14, 14), mode='stretch', fill='mean', invalidates=None) invalidates statistics requested by Outliers: sharpness, offset_x, offset_y, width, height, size, aspect_ratio, depth, center, distance_center, distance_edge, invalid_box. These now describe the transform rather than the source data. If this is model preprocessing, move it to the extractor's transforms=; otherwise pass flags= to exclude them.


In [8]:
# TEST ASSERTION CELL ###
assert len(invalidated) == 1
assert "sharpness" in str(invalidated[0].message)

In [9]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    Outliers(flags=ImageStats.PIXEL).evaluate(deployed)

print("warnings:", [w.category.__name__ for w in caught if "Invalidated" in w.category.__name__])

warnings: []


In [10]:
asserted = View(mnist, [Resize((14, 14), invalidates=ImageStats.VISUAL_SHARPNESS)])

warned = {}
for name, flags in [("DIMENSION", ImageStats.DIMENSION), ("VISUAL", ImageStats.VISUAL)]:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        Outliers(flags=flags).evaluate(asserted)
    warned[name] = any(w.category.__name__ == "StatsInvalidatedWarning" for w in caught)
    print(f"{name:>9}: {'warned' if warned[name] else 'silent'}")

DIMENSION: silent


   VISUAL: warned


In [11]:
# TEST ASSERTION CELL ###
assert warned == {"DIMENSION": False, "VISUAL": True}

In [12]:
model = torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(14 * 14, 16))


def normalize(image: torch.Tensor) -> torch.Tensor:
    """Model preprocessing: scale to [0, 1], then standardize."""
    return (image / 255.0 - 0.1307) / 0.3081


extractor = TorchExtractor(model, transforms=normalize)
embedding = extractor(np.asarray(deployed[0][0])[None])

print("view sees:", np.asarray(deployed[0][0]).shape, "-- raw pixel values")
print("extractor produces:", tuple(np.asarray(embedding).shape), "-- normalized, model-side only")

view sees: (1, 14, 14) -- raw pixel values
extractor produces: (1, 16) -- normalized, model-side only


In [13]:
# TEST ASSERTION CELL ###
assert np.asarray(embedding).shape[-1] == 16

In [14]:
print("as a claim about the data:", np.asarray(View(mnist, [SelectChannels("rgb")])[0][0]).shape)
print("...but ask whether the data really has three channels")

as a claim about the data: (3, 28, 28)
...but ask whether the data really has three channels
